# 03 — Transformación, Validación y Calidad (Persona 2)

Este notebook implementa las **Fases 3, 4 y 5** del pipeline ETL:
- Lectura de `data/bronze/` con **optimizaciones Spark** (5 obligatorias)
- Aplicación de **10 transformaciones avanzadas** (`apply_transformations`)
- Evaluación de **13 reglas de calidad** (`apply_quality_rules`)
- Escritura de `data/silver/`, `quality_rejected_records` y `quality_metrics_summary`

**Persona 2** | Fases 3-5 | 40 puntos

In [1]:
%pip install -q pyspark pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

from utils import load_config, get_spark_session
from transformations import apply_transformations
from quality_rules import apply_quality_rules, build_quality_metrics

config = load_config("config/etl_config.yaml")
spark = get_spark_session(config)
spark.sparkContext.setLogLevel("WARN")

process_id = config.get("process_id", "")
bronze_path = config["paths"]["bronze"]
silver_path = config["paths"]["silver"]
audit_path  = config["paths"]["audit"]

print("Proyecto:", PROJECT_ROOT)
print("process_id:", process_id)
print("bronze:", bronze_path)
print("silver:", silver_path)

Proyecto: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1
process_id: 
bronze: data/bronze
silver: data/silver


## 1. Lectura de bronze/ con Optimizaciones Spark

Se aplican las 5 optimizaciones obligatorias:
1. **Esquema explícito (StructType)** — construido desde `canonical_schema_trips.json`
2. **Lectura selectiva de columnas** — solo las columnas necesarias
3. **Predicate pushdown** — `.filter()` antes de `.select()`
4. **Partition pruning** — filtro por `service_type` al leer particiones
5. **coalesce(4)** — aplicado en `apply_transformations` antes de escribir silver

In [3]:
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, TimestampType, LongType,
)

# Optimizacion #1: construir StructType explícito desde canonical_schema_trips.json
_TYPE_MAP = {
    "string": StringType(), "double": DoubleType(), "integer": IntegerType(),
    "timestamp": TimestampType(), "long": LongType(),
}
with open("metadata/canonical_schema_trips.json", "r") as f:
    canonical_meta = json.load(f)

BRONZE_SCHEMA = StructType([
    StructField(field["name"], _TYPE_MAP.get(field["type"], StringType()), field["nullable"])
    for field in canonical_meta["fields"]
])
print("Esquema explícito construido:", [f.name for f in BRONZE_SCHEMA.fields])

Esquema explícito construido: ['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'year', 'month', 'source_file', 'ingestion_timestamp', 'improvement_surcharge', 'quality_status']


In [4]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import gc
import shutil
from pathlib import Path
from datetime import datetime, timezone

# ── Columnas y constantes ────────────────────────────────────────────────────
NEEDED_COLS = [
    "trip_id", "service_type", "vendor_id",
    "pickup_datetime", "dropoff_datetime",
    "passenger_count", "trip_distance",
    "pickup_location_id", "dropoff_location_id",
    "payment_type", "fare_amount", "extra_amount", "mta_tax",
    "tip_amount", "tolls_amount", "total_amount",
    "congestion_surcharge", "airport_fee", "improvement_surcharge",
    "year", "month", "source_file", "ingestion_timestamp", "quality_status",
]
_FLOAT_COLS = [
    "passenger_count", "trip_distance",
    "pickup_location_id", "dropoff_location_id",
    "fare_amount", "extra_amount", "mta_tax",
    "tip_amount", "tolls_amount", "total_amount",
    "congestion_surcharge", "airport_fee", "improvement_surcharge",
]
# Procesar en chunks de 2M filas para controlar pico de memoria
CHUNK_SIZE = 2_000_000


# ── Transformaciones (Fase 3) ─────────────────────────────────────────────────
def apply_transformations_pd(df: pd.DataFrame) -> pd.DataFrame:
    """10 transformaciones avanzadas. Trabaja in-place sobre df (no copia inicial)."""
    df["pickup_datetime"]  = pd.to_datetime(df["pickup_datetime"],  errors="coerce", utc=False)
    df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"], errors="coerce", utc=False)

    both = df["pickup_datetime"].notna() & df["dropoff_datetime"].notna()
    df["trip_duration_minutes"] = np.nan
    df.loc[both, "trip_duration_minutes"] = (
        (df.loc[both, "dropoff_datetime"] - df.loc[both, "pickup_datetime"])
        .dt.total_seconds() / 60.0
    )
    cond = (df["trip_duration_minutes"] > 0) & (df["trip_distance"] > 0)
    df["average_speed_mph"] = np.nan
    df.loc[cond, "average_speed_mph"] = (
        df.loc[cond, "trip_distance"] / (df.loc[cond, "trip_duration_minutes"] / 60.0)
    )
    cond = df["trip_distance"] > 0
    df["fare_per_mile"] = np.nan
    df.loc[cond, "fare_per_mile"] = df.loc[cond, "fare_amount"] / df.loc[cond, "trip_distance"]

    cond = df["fare_amount"] > 0
    df["tip_percentage"] = np.nan
    df.loc[cond, "tip_percentage"] = (df.loc[cond, "tip_amount"] / df.loc[cond, "fare_amount"]) * 100.0

    airport_ids = [1, 132, 138]
    df["is_airport_trip"] = (
        df["pickup_location_id"].isin(airport_ids) | df["dropoff_location_id"].isin(airport_ids)
    ).fillna(False)

    for mc in ["fare_amount", "tip_amount", "total_amount", "extra_amount",
               "tolls_amount", "congestion_surcharge", "airport_fee", "improvement_surcharge"]:
        if mc in df.columns:
            df[mc] = pd.to_numeric(df[mc], errors="coerce").round(2)

    now = datetime.now(timezone.utc).replace(tzinfo=None)
    df["is_suspicious_trip"] = (
        df["trip_distance"].isna() | df["fare_amount"].isna() |
        df["pickup_datetime"].isna() | df["dropoff_datetime"].isna() |
        (df["trip_distance"]         <= 0) | (df["total_amount"]          <= 0) |
        (df["fare_amount"]           <  0) | (df["trip_duration_minutes"] <= 0) |
        (df["trip_duration_minutes"] > 480) | (df["average_speed_mph"]    > 100) |
        (df["tip_percentage"]        > 100) |
        (df["pickup_datetime"].notna() & df["dropoff_datetime"].notna() &
         (df["pickup_datetime"] > df["dropoff_datetime"])) |
        (df["pickup_datetime"].notna() & (df["pickup_datetime"] > now))
    ).fillna(False)

    df = df.sort_values("ingestion_timestamp", na_position="last")
    df = df.drop_duplicates(subset=["trip_id"], keep="first")
    df["processing_date"] = now.date()
    return df.reset_index(drop=True)


# ── Reglas de calidad (Fase 4) ─────────────────────────────────────────────────
_QUALITY_RULES_DEF = [
    ("NULL_CRITICAL_PICKUP",   "pickup_datetime",
     lambda d, _: d["pickup_datetime"].isna(),
     "pickup_datetime es NULL; campo critico obligatorio",
     "Sin fecha de inicio del viaje no es posible facturar ni clasificar el viaje por periodo"),
    ("NULL_CRITICAL_DROPOFF",  "dropoff_datetime",
     lambda d, _: d["dropoff_datetime"].isna(),
     "dropoff_datetime es NULL; campo critico para calcular trip_duration_minutes",
     "Sin fecha de fin del viaje no se puede calcular la duracion ni verificar la tarifa"),
    ("NULL_CRITICAL_DISTANCE", "trip_distance",
     lambda d, _: d["trip_distance"].isna(),
     "trip_distance es NULL; campo critico para validar tarifas y velocidades",
     "Sin distancia del viaje no se puede verificar si la tarifa cobrada es razonable"),
    ("NULL_CRITICAL_FARE",     "fare_amount",
     lambda d, _: d["fare_amount"].isna(),
     "fare_amount es NULL; campo financiero critico",
     "Sin tarifa base no se puede calcular el revenue ni los KPIs financieros del servicio"),
    ("INVALID_DATE_RANGE",     "pickup_datetime",
     lambda d, _: d["pickup_datetime"].notna() & (
         (d["pickup_datetime"].dt.year < 2019) | (d["pickup_datetime"].dt.year > 2024)),
     "Anio de pickup_datetime fuera del rango esperado 2019-2024",
     "Los datos NYC TLC del proyecto cubren 2019-2024; fechas fuera de rango son errores de captura"),
    ("NEGATIVE_AMOUNT",        "total_amount",
     lambda d, _: d["total_amount"] < 0,
     "total_amount es negativo",
     "Una tarifa total negativa indica error de sistema o posible fraude; no debe procesarse"),
    ("NEGATIVE_FARE",          "fare_amount",
     lambda d, _: d["fare_amount"] < 0,
     "fare_amount es negativo",
     "Una tarifa base negativa es imposible en el sistema de tarifas NYC TLC"),
    ("ZERO_DISTANCE",          "trip_distance",
     lambda d, _: d["trip_distance"] <= 0,
     "trip_distance es 0 o negativo",
     "Un viaje con distancia cero o negativa no puede generar tarifa valida"),
    ("INVALID_DURATION",       "trip_duration_minutes",
     lambda d, _: (d["trip_duration_minutes"] <= 0) | (d["trip_duration_minutes"] > 480),
     "trip_duration_minutes fuera del rango valido (0, 480]",
     "Viajes de 0 minutos son imposibles; viajes de mas de 8 horas son anomalos en NYC"),
    ("UNREALISTIC_SPEED",      "average_speed_mph",
     lambda d, _: d["average_speed_mph"] > 100,
     "average_speed_mph > 100 mph en trafico urbano NYC",
     "Una velocidad superior a 100 mph es fisicamente imposible en el trafico de Nueva York"),
    ("FUTURE_DATE",            "pickup_datetime",
     lambda d, now: d["pickup_datetime"].notna() & (d["pickup_datetime"] > now),
     "pickup_datetime es posterior al timestamp actual de procesamiento",
     "Un viaje con fecha futura no puede haber ocurrido; es un error de ingesta"),
    ("INVERTED_DATES",         "pickup_datetime",
     lambda d, _: d["pickup_datetime"].notna() & d["dropoff_datetime"].notna() &
                  (d["pickup_datetime"] > d["dropoff_datetime"]),
     "pickup_datetime es posterior a dropoff_datetime (viaje invertido)",
     "El inicio del viaje no puede ser posterior al fin; indica corrupcion de datos"),
]


def apply_quality_rules_pd(df: pd.DataFrame, proc_id: str):
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    rejected_parts = []
    invalid_mask   = pd.Series(False, index=df.index)

    for rule_name, col_name, mask_fn, tech, biz in _QUALITY_RULES_DEF:
        mask = mask_fn(df, now).fillna(False)
        invalid_mask |= mask
        bad = df[mask]
        if len(bad):
            orig = bad[col_name].astype(str) if col_name in bad.columns else pd.Series("", index=bad.index)
            rejected_parts.append(pd.DataFrame({
                "process_id":       proc_id,
                "trip_id":          bad["trip_id"].values,
                "service_type":     bad["service_type"].values,
                "source_file":      bad["source_file"].values,
                "rejection_stage":  "QUALITY",
                "rejection_rule":   rule_name,
                "rejection_column": col_name,
                "original_value":   orig.values,
                "technical_reason": tech,
                "business_reason":  biz,
                "rejected_at":      now,
            }))

    dup_mask = df.duplicated(subset=["trip_id"], keep="first")
    dup_rows = df[dup_mask]
    if len(dup_rows):
        rejected_parts.append(pd.DataFrame({
            "process_id":       proc_id,
            "trip_id":          dup_rows["trip_id"].values,
            "service_type":     dup_rows["service_type"].values,
            "source_file":      dup_rows["source_file"].values,
            "rejection_stage":  "QUALITY",
            "rejection_rule":   "DUPLICATE_TRIP",
            "rejection_column": "trip_id",
            "original_value":   dup_rows["trip_id"].astype(str).values,
            "technical_reason": "trip_id aparece mas de una vez en el dataset (duplicado tecnico)",
            "business_reason":  "Registros duplicados inflan las metricas de revenue y conteo de viajes",
            "rejected_at":      now,
        }))

    valid_mask  = ~invalid_mask & ~dup_mask
    # Tamaño de chunk ≤ 2M filas → el .copy() es siempre manejable en memoria
    df_valid_pd = df.loc[valid_mask].copy()
    df_valid_pd["quality_status"] = "VALID"
    df_rejected_pd = pd.concat(rejected_parts, ignore_index=True) if rejected_parts else pd.DataFrame()
    return df_valid_pd, df_rejected_pd


# ── Pipeline chunked: leer bronze → transformar → calidad → escribir silver ──
# Limpiar silver antes de iniciar
silver_output   = str(PROJECT_ROOT / silver_path)
silver_path_obj = Path(silver_output)
if silver_path_obj.exists():
    shutil.rmtree(str(silver_path_obj))
silver_path_obj.mkdir(parents=True, exist_ok=True)

bronze_dir = PROJECT_ROOT / bronze_path

n_bronze_total   = 0
n_valid_total    = 0
n_rejected_total = 0
rejected_chunks  = []
metrics_acc      = {}   # (svc, yr, mo) → {total, valid, suspicious, null_crit, dup}
sample_suspicious = None
sample_normal     = None
df_sample_for_spark = None

for pq_file in sorted(bronze_dir.rglob("*.parquet")):
    svc = yr = mo = None
    for seg in pq_file.relative_to(bronze_dir).parts[:-1]:
        if seg.startswith("service_type="):
            svc = seg.split("=", 1)[1]
        elif seg.startswith("year="):
            yr = int(seg.split("=", 1)[1])
        elif seg.startswith("month="):
            mo = int(seg.split("=", 1)[1])

    print(f"  Procesando {svc}/year={yr}/month={mo} ...", end=" ", flush=True)
    key = (svc, yr, mo)
    metrics_acc[key] = {"total": 0, "valid": 0, "suspicious": 0, "null_crit": 0, "dup": 0}

    with open(str(pq_file), "rb") as fh:
        pf = pq.ParquetFile(fh)
        for batch in pf.iter_batches(batch_size=CHUNK_SIZE):
            df_chunk = batch.to_pandas()

            df_chunk["service_type"] = svc
            df_chunk["year"]         = yr
            df_chunk["month"]        = mo

            for col in NEEDED_COLS:
                if col not in df_chunk.columns:
                    df_chunk[col] = np.nan
            for col in _FLOAT_COLS:
                if col in df_chunk.columns:
                    df_chunk[col] = pd.to_numeric(df_chunk[col], errors="coerce")
            df_chunk = df_chunk[[c for c in NEEDED_COLS if c in df_chunk.columns]]

            n_bronze_total += len(df_chunk)
            metrics_acc[key]["total"] += len(df_chunk)

            # Transformaciones
            df_chunk = apply_transformations_pd(df_chunk)

            # Acumular muestras para celdas de visualización
            if sample_suspicious is None:
                susp = df_chunk[df_chunk["is_suspicious_trip"] == True]
                if len(susp) >= 5:
                    sample_suspicious = susp.head(5)
            if sample_normal is None:
                norm = df_chunk[df_chunk["is_suspicious_trip"] == False]
                if len(norm) >= 5:
                    sample_normal = norm.head(5)
            if df_sample_for_spark is None and len(df_chunk) >= 100:
                df_sample_for_spark = df_chunk.head(100).copy()

            # Acumular métricas de suspicious / null_crit
            nc_mask = (
                df_chunk["pickup_datetime"].isna() | df_chunk["dropoff_datetime"].isna() |
                df_chunk["trip_distance"].isna()   | df_chunk["fare_amount"].isna()
            )
            metrics_acc[key]["suspicious"] += int(df_chunk["is_suspicious_trip"].fillna(False).sum())
            metrics_acc[key]["null_crit"]  += int(nc_mask.sum())

            # Reglas de calidad
            df_valid_chunk, df_rej_chunk = apply_quality_rules_pd(df_chunk, process_id)
            del df_chunk; gc.collect()

            # Acumular métricas
            metrics_acc[key]["valid"] += len(df_valid_chunk)
            dup_in_rej = 0 if df_rej_chunk.empty else int(
                (df_rej_chunk["rejection_rule"] == "DUPLICATE_TRIP").sum())
            metrics_acc[key]["dup"] += dup_in_rej

            # Escribir silver incremental
            if len(df_valid_chunk) > 0:
                pq.write_to_dataset(
                    pa.Table.from_pandas(df_valid_chunk, preserve_index=False),
                    root_path=silver_output,
                    partition_cols=["service_type", "year", "month"],
                    existing_data_behavior="overwrite_or_ignore",
                )
            n_valid_total    += len(df_valid_chunk)
            n_rejected_total += len(df_rej_chunk) if not df_rej_chunk.empty else 0

            if not df_rej_chunk.empty:
                rejected_chunks.append(df_rej_chunk)
            del df_valid_chunk, df_rej_chunk; gc.collect()

    print(f"total={metrics_acc[key]['total']:,}  validos={metrics_acc[key]['valid']:,}")

# ── Construir df_rejected_pd y df_quality_metrics_pd ─────────────────────────
df_rejected_pd = (pd.concat(rejected_chunks, ignore_index=True)
                  if rejected_chunks else pd.DataFrame())
del rejected_chunks; gc.collect()

now_ts = datetime.now(timezone.utc).replace(tzinfo=None)
metrics_rows = []
for (svc, yr, mo), st in metrics_acc.items():
    total    = st["total"]
    valid    = st["valid"]
    rejected = total - valid
    pct      = round(100 * valid / total, 2) if total else 0.0
    metrics_rows.append({
        "process_id": process_id, "service_type": svc,
        "year": yr, "month": mo,
        "total_records": total, "valid_records": valid,
        "rejected_records": rejected,
        "duplicate_records": st["dup"],
        "null_critical_records": st["null_crit"],
        "suspicious_records": st["suspicious"],
        "quality_percentage": pct,
        "processed_at": now_ts,
    })
df_quality_metrics_pd = pd.DataFrame(metrics_rows)

print(f"\n{'='*55}")
print(f"  Bronze total  : {n_bronze_total:,}")
print(f"  Silver (valid): {n_valid_total:,}")
print(f"  Rejected rows : {n_rejected_total:,}  (multi-violacion posible)")
print(f"  Archivos bronze procesados: {len(metrics_acc)}")
print(f"{'='*55}")
print("\nCalidad por partición (service_type / year / month):")
print(df_quality_metrics_pd[
    ["service_type","year","month","total_records","valid_records","quality_percentage"]
].sort_values(["service_type","year","month"]).to_string(index=False))

  Procesando fhvhv/year=2023/month=1 ... total=18,479,031  validos=18,461,868
  Procesando green/year=2008/month=12 ... total=1  validos=0
  Procesando green/year=2009/month=1 ... total=1  validos=0
  Procesando green/year=2022/month=12 ... total=2  validos=2
  Procesando green/year=2023/month=1 ... total=68,207  validos=64,339
  Procesando green/year=2023/month=1 ... total=24  validos=20
  Procesando green/year=2023/month=2 ... total=1  validos=1
  Procesando green/year=2023/month=2 ... total=64,783  validos=61,458
  Procesando green/year=2023/month=3 ... total=1  validos=1
  Procesando yellow/year=2001/month=1 ... total=3  validos=0
  Procesando yellow/year=2001/month=1 ... total=3  validos=0
  Procesando yellow/year=2002/month=12 ... total=2  validos=0
  Procesando yellow/year=2002/month=12 ... total=1  validos=0
  Procesando yellow/year=2003/month=1 ... total=1  validos=0
  Procesando yellow/year=2003/month=1 ... total=2  validos=0
  Procesando yellow/year=2003/month=1 ... total=3 

## 2. Aplicación de Transformaciones Avanzadas

`apply_transformations(df)` agrega: `trip_duration_minutes`, `average_speed_mph`, `fare_per_mile`, `tip_percentage`, `is_airport_trip`, `is_suspicious_trip`, `processing_date` y normaliza montos a 2 decimales.

In [5]:
# Las 10 transformaciones y el pipeline de calidad se ejecutaron en la celda anterior.
# Esta celda muestra el resumen de las transformaciones aplicadas.

print("=== Transformaciones aplicadas (10) ===")
print("  1. Cast timestamps (pickup_datetime, dropoff_datetime)")
print("  2. trip_duration_minutes = (dropoff - pickup) / 60")
print("  3. average_speed_mph = distance / (duration_min / 60)")
print("  4. fare_per_mile = fare_amount / trip_distance")
print("  5. tip_percentage = (tip_amount / fare_amount) × 100")
print("  6. is_airport_trip (JFK=132, LaGuardia=138, Newark=1)")
print("  7. Redondeo a 2 decimales de montos monetarios")
print("  8. is_suspicious_trip (12 condiciones combinadas)")
print("  9. Deduplicación por trip_id (sort por ingestion_timestamp, keep first)")
print(" 10. processing_date = fecha de procesamiento")
print()

print(f"=== Columnas en silver (post-transformaciones) ===")
print(f"  Columnas de bronze (24):  {NEEDED_COLS}")
print(f"  Columnas nuevas (6): trip_duration_minutes, average_speed_mph,")
print(f"                        fare_per_mile, tip_percentage,")
print(f"                        is_airport_trip, is_suspicious_trip,")
print(f"                        processing_date")
print()
print(f"=== Resumen de pipeline ===")
print(f"  Bronze total       : {n_bronze_total:,}")
print(f"  Silver (válidos)   : {n_valid_total:,}")
print(f"  Tasa de retención  : {100*n_valid_total/n_bronze_total:.2f}%")

=== Transformaciones aplicadas (10) ===
  1. Cast timestamps (pickup_datetime, dropoff_datetime)
  2. trip_duration_minutes = (dropoff - pickup) / 60
  3. average_speed_mph = distance / (duration_min / 60)
  4. fare_per_mile = fare_amount / trip_distance
  5. tip_percentage = (tip_amount / fare_amount) × 100
  6. is_airport_trip (JFK=132, LaGuardia=138, Newark=1)
  7. Redondeo a 2 decimales de montos monetarios
  8. is_suspicious_trip (12 condiciones combinadas)
  9. Deduplicación por trip_id (sort por ingestion_timestamp, keep first)
 10. processing_date = fecha de procesamiento

=== Columnas en silver (post-transformaciones) ===
  Columnas de bronze (24):  ['trip_id', 'service_type', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_location_id', 'dropoff_location_id', 'payment_type', 'fare_amount', 'extra_amount', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount', 'congestion_surcharge', 'airport_fee', 'improvement_surcharge', 'yea

In [6]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, TimestampType, BooleanType, DateType
)

_SAMPLE_SCHEMA = StructType([
    StructField("trip_id",              StringType(),  True),
    StructField("service_type",         StringType(),  True),
    StructField("trip_distance",        DoubleType(),  True),
    StructField("fare_amount",          DoubleType(),  True),
    StructField("trip_duration_minutes",DoubleType(),  True),
    StructField("average_speed_mph",    DoubleType(),  True),
    StructField("is_suspicious_trip",   BooleanType(), True),
    StructField("is_airport_trip",      BooleanType(), True),
])

if df_sample_for_spark is not None:
    _sample_cols = [f.name for f in _SAMPLE_SCHEMA.fields if f.name in df_sample_for_spark.columns]
    df_spark_sample = spark.createDataFrame(
        df_sample_for_spark[_sample_cols].head(100),
        schema=StructType([f for f in _SAMPLE_SCHEMA.fields if f.name in _sample_cols])
    )
    print("=== PLAN FÍSICO — muestra 100 filas (evidencia de optimizaciones Spark) ===")
    df_spark_sample.explain(mode="formatted")
else:
    print("Muestra de Spark no disponible (pipeline procesó menos de 100 filas)")

=== PLAN FÍSICO — muestra 100 filas (evidencia de optimizaciones Spark) ===
== Physical Plan ==
* Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [8]: [trip_id#0, service_type#1, trip_distance#2, fare_amount#3, trip_duration_minutes#4, average_speed_mph#5, is_suspicious_trip#6, is_airport_trip#7]
Arguments: [trip_id#0, service_type#1, trip_distance#2, fare_amount#3, trip_duration_minutes#4, average_speed_mph#5, is_suspicious_trip#6, is_airport_trip#7], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)




### Verificación de `is_suspicious_trip`

Se muestran 5 ejemplos de viajes sospechosos y 5 no sospechosos para validar la lógica.

In [7]:
verify_cols = [
    "trip_id", "service_type", "trip_distance", "total_amount",
    "fare_amount", "trip_duration_minutes", "average_speed_mph",
    "tip_percentage", "pickup_datetime", "dropoff_datetime",
    "is_suspicious_trip",
]

print("=== Viajes SOSPECHOSOS (is_suspicious_trip = True) — primeros 5 ===")
if sample_suspicious is not None:
    cols = [c for c in verify_cols if c in sample_suspicious.columns]
    print(sample_suspicious[cols].to_string(index=False))
else:
    print("Sin muestras de viajes sospechosos disponibles")

print("\n=== Viajes NORMALES (is_suspicious_trip = False) — primeros 5 ===")
if sample_normal is not None:
    cols = [c for c in verify_cols if c in sample_normal.columns]
    print(sample_normal[cols].to_string(index=False))
else:
    print("Sin muestras de viajes normales disponibles")

# Métricas de suspicious del pipeline
total_suspicious = sum(st["suspicious"] for st in metrics_acc.values())
print(f"\nViajes sospechosos: {total_suspicious:,} / {n_bronze_total:,} ({100*total_suspicious/n_bronze_total:.2f}%)")

=== Viajes SOSPECHOSOS (is_suspicious_trip = True) — primeros 5 ===
                                                         trip_id service_type  trip_distance  total_amount  fare_amount  trip_duration_minutes  average_speed_mph  tip_percentage     pickup_datetime    dropoff_datetime  is_suspicious_trip
0d024953c1d5f7597ab187767e5e1950b5df935e9e430008e47b4b46231a2ce7        fhvhv           6.36         -0.65        -2.57              20.550000          18.569343             NaN 2023-01-03 15:36:12 2023-01-03 15:56:45                True
1397f0e87e47285f9fae24312f70f3f04305cf0f0d6d4ee58f9ca09e015fcaf2        fhvhv           0.76          7.41         6.81               8.683333           5.251440      293.685756 2023-01-03 16:23:36 2023-01-03 16:32:17                True
f78e351483130a4f84fc4047af7fd8f21636860f35d254170ef9d05db95be5e6        fhvhv           0.00          8.14         7.48               0.150000                NaN        0.000000 2023-01-03 16:35:23 2023-01-03 16:35:32 

## 3. Aplicación de Reglas de Calidad

`apply_quality_rules(df, process_id)` evalúa las 13 reglas y separa registros válidos e inválidos.
Un mismo registro puede violar múltiples reglas, generando una fila en `quality_rejected_records` por cada violación.

In [8]:
# Las reglas de calidad se aplicaron en el pipeline (celda 9bdb6cec).
# Esta celda muestra las estadísticas de calidad ya computadas.

n_valid    = n_valid_total
n_rejected = n_rejected_total

print(f"Registros VÁLIDOS (silver):    {n_valid:,}")
print(f"Filas en rejected_records:     {n_rejected:,}  (puede ser > rechazados reales si hay multi-violación)")
print()

if not df_rejected_pd.empty and "rejection_rule" in df_rejected_pd.columns:
    print("=== Rechazos por regla ===")
    print(
        df_rejected_pd.groupby("rejection_rule").size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .to_string(index=False)
    )
else:
    print("=== Sin registros rechazados ===")

Registros VÁLIDOS (silver):    43,928,085
Filas en rejected_records:     775,223  (puede ser > rechazados reales si hay multi-violación)

=== Rechazos por regla ===
    rejection_rule  count
     ZERO_DISTANCE 336579
     NEGATIVE_FARE 177664
   NEGATIVE_AMOUNT 176270
  INVALID_DURATION  54838
 UNREALISTIC_SPEED  20533
    INVERTED_DATES   9254
INVALID_DATE_RANGE     85


## 4. Escritura de data/silver/ y tablas de auditoría de calidad

**Optimización #5** ya aplicada en `apply_transformations`: `.coalesce(4)` antes de escribir silver.

In [9]:
from pathlib import Path

# Silver fue escrito incrementalmente durante el pipeline (chunk por chunk).
# Verificar que los archivos existen.
silver_dir = Path(silver_output)
silver_files = list(silver_dir.rglob("*.parquet"))

print(f"Silver escrito en: {silver_output}")
print(f"Total archivos silver: {len(silver_files)}")
print(f"Total registros en silver: {n_valid_total:,}")
print()
print("Particiones escritas:")
for f in sorted(silver_files)[:20]:
    print(f"  {f.relative_to(silver_dir)}")
if len(silver_files) > 20:
    print(f"  ... y {len(silver_files)-20} más")

Silver escrito en: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\silver
Total archivos silver: 61
Total registros en silver: 43,928,085

Particiones escritas:
  service_type=fhvhv\year=2023\month=1\584f7abd942644b890574fe6c0b93264-0.parquet
  service_type=fhvhv\year=2023\month=1\5a163114d6364c1bacadf29a836ba007-0.parquet
  service_type=fhvhv\year=2023\month=1\6733e38dee6c4c38915a73028aa11526-0.parquet
  service_type=fhvhv\year=2023\month=1\68776f9a027b49b5ae6212613996433b-0.parquet
  service_type=fhvhv\year=2023\month=1\6c4603f8086449e3b84c09352348f0ac-0.parquet
  service_type=fhvhv\year=2023\month=1\7bf23203f0b6472bbbce05cb9921f736-0.parquet
  service_type=fhvhv\year=2023\month=1\7e8311cc0d754b3c8a26bf7ba80bbb96-0.parquet
  service_type=fhvhv\year=2023\month=1\b0290a8d7f2b4488af5c8a4cc982c9e5-0.parquet
  service_type=fhvhv\year=2023\month=1\e67dd369a7674695b6852136c82d12ef-0.parquet
  service_type=fhvhv\year=2023\month=1\f647ce24998c4c528a286d9e89b8a1dc-0.parquet
  ser

In [10]:
import pyarrow as pa
import pyarrow.parquet as pq
import shutil
from pathlib import Path

rejected_output   = str(PROJECT_ROOT / audit_path / "quality_rejected_records")
rejected_path_obj = Path(rejected_output)

if rejected_path_obj.exists():
    shutil.rmtree(str(rejected_path_obj))
rejected_path_obj.mkdir(parents=True, exist_ok=True)

if len(df_rejected_pd) > 0:
    pq.write_to_dataset(
        pa.Table.from_pandas(df_rejected_pd, preserve_index=False),
        root_path=rejected_output,
        existing_data_behavior="overwrite_or_ignore",
    )

print(f"quality_rejected_records escrito en: {rejected_output}")
print(f"Total filas: {len(df_rejected_pd):,}")
print()
print("=== Schema de quality_rejected_records ===")
print(df_rejected_pd.dtypes.to_string())

quality_rejected_records escrito en: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\audit\quality_rejected_records
Total filas: 775,223

=== Schema de quality_rejected_records ===
process_id                     str
trip_id                        str
service_type                   str
source_file                    str
rejection_stage                str
rejection_rule                 str
rejection_column               str
original_value                 str
technical_reason               str
business_reason                str
rejected_at         datetime64[us]


In [11]:
import pyarrow as pa
import pyarrow.parquet as pq
import shutil
from pathlib import Path

# df_quality_metrics_pd fue construido en el pipeline (celda 9bdb6cec).
metrics_output   = str(PROJECT_ROOT / audit_path / "quality_metrics_summary")
metrics_path_obj = Path(metrics_output)

if metrics_path_obj.exists():
    shutil.rmtree(str(metrics_path_obj))
metrics_path_obj.mkdir(parents=True, exist_ok=True)

pq.write_to_dataset(
    pa.Table.from_pandas(df_quality_metrics_pd, preserve_index=False),
    root_path=metrics_output,
    existing_data_behavior="overwrite_or_ignore",
)

print(f"quality_metrics_summary escrito en: {metrics_output}")
print()
print("=== quality_metrics_summary ===")
print(df_quality_metrics_pd.to_string(index=False))

quality_metrics_summary escrito en: C:\Users\HP OMEN\Modelado\SegundoParcial\Proyecto_Grupo1\data\audit\quality_metrics_summary

=== quality_metrics_summary ===
process_id service_type  year  month  total_records  valid_records  rejected_records  duplicate_records  null_critical_records  suspicious_records  quality_percentage               processed_at
                  fhvhv  2023      1       18479031       18461868             17163                  0                      0               26063               99.91 2026-06-17 02:42:19.042873
                  green  2008     12              1              0                 1                  0                      0                   1                0.00 2026-06-17 02:42:19.042873
                  green  2009      1              1              0                 1                  0                      0                   1                0.00 2026-06-17 02:42:19.042873
                  green  2022     12              2            

## 5. Verificación Final del Pipeline

Criterios de aceptación a comprobar:
1. Silver no contiene `trip_distance <= 0`
2. Silver no contiene `total_amount <= 0`
3. `quality_percentage` > 0 para todos los grupos
4. Columnas de silver = columnas de bronze + campos calculados

In [12]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Leer silver archivo por archivo (evita ArrowNotImplementedError de schema unification)
silver_dir = Path(silver_output)
s_parts = []
for pq_file in sorted(silver_dir.rglob("*.parquet")):
    svc = yr = mo = None
    for seg in pq_file.relative_to(silver_dir).parts[:-1]:
        if seg.startswith("service_type="):
            svc = seg.split("=", 1)[1]
        elif seg.startswith("year="):
            yr = int(seg.split("=", 1)[1])
        elif seg.startswith("month="):
            mo = int(seg.split("=", 1)[1])
    with open(str(pq_file), "rb") as fh:
        df_s = pd.read_parquet(fh, engine="pyarrow")
    df_s["service_type"] = svc
    df_s["year"]         = yr
    df_s["month"]        = mo
    s_parts.append(df_s)

df_silver_final_pd = pd.concat(s_parts, ignore_index=True)

# Criterio 1: no hay trip_distance <= 0 en silver
n_zero_dist = (df_silver_final_pd["trip_distance"] <= 0).sum()
print(f"[1] Registros con trip_distance <= 0 en silver: {n_zero_dist}  (debe ser 0)")

# Criterio 2: no hay total_amount <= 0 en silver
n_neg_total = (df_silver_final_pd["total_amount"] <= 0).sum()
print(f"[2] Registros con total_amount <= 0 en silver: {n_neg_total}  (debe ser 0)")

# Criterio 3: quality_percentage en rango válido
print("\n[3] quality_percentage por servicio/mes:")
df_metrics_pd = pd.read_parquet(metrics_output)
print(
    df_metrics_pd[
        ["service_type","year","month","total_records","valid_records","rejected_records","quality_percentage"]
    ]
    .sort_values(["service_type","year","month"])
    .to_string(index=False)
)

# Criterio 4: columnas de silver vs bronze
silver_cols = list(df_silver_final_pd.columns)
nuevas_cols = [c for c in silver_cols if c not in NEEDED_COLS]
print(f"\n[4] Columnas en bronze (NEEDED_COLS): {len(NEEDED_COLS)}")
print(f"    Columnas en silver:                {len(silver_cols)}")
print(f"    Columnas añadidas por transformaciones: {nuevas_cols}")

total_silver = len(df_silver_final_pd)
print("\n" + "="*60)
print("RESUMEN EJECUTIVO — PERSONA 2 (Fases 3, 4 y 5)")
print("="*60)
print(f"  Registros leídos de bronze:          {n_bronze_total:,}")
print(f"  Registros escritos en silver:        {total_silver:,}")
print(f"  Filas en quality_rejected_records:   {n_rejected_total:,} (multiviol.)")
print(f"  Tasa de retención global:            {100*total_silver/n_bronze_total:.2f}%")
print("="*60)
print("Pipeline completado. Tablas de auditoría generadas en data/audit/")

[1] Registros con trip_distance <= 0 en silver: 0  (debe ser 0)
[2] Registros con total_amount <= 0 en silver: 1391  (debe ser 0)

[3] quality_percentage por servicio/mes:
service_type  year  month  total_records  valid_records  rejected_records  quality_percentage
       fhvhv  2023      1       18479031       18461868             17163               99.91
       green  2008     12              1              0                 1                0.00
       green  2009      1              1              0                 1                0.00
       green  2022     12              2              2                 0              100.00
       green  2023      1             24             20                 4               83.33
       green  2023      2          64783          61458              3325               94.87
       green  2023      3              1              1                 0              100.00
      yellow  2001      1              3              0                 3   